# Pattern — Evaluator-Optimizer

The **evaluator-optimizer** pattern runs a **generate → evaluate → revise** loop. One component produces a draft; a separate **evaluator** scores it against criteria and returns actionable feedback; the **optimizer** (often the same generator with revision instructions) improves the draft until quality passes or a budget is exhausted.

```
  Request + context
        │
        ▼
   ┌──────────┐
   │ GENERATE │  draft v1
   └────┬─────┘
        ▼
   ┌──────────┐
   │ EVALUATE │  score + feedback
   └────┬─────┘
        │ pass? ──yes──► final output
        │ no
        ▼
   ┌──────────┐
   │ OPTIMIZE │  draft v2 … vN
   └────┬─────┘
        └──► loop (max iterations)
```

Unlike a fixed **prompt chain**, the loop is **conditional** — you only pay for extra iterations when the evaluator finds gaps.

This notebook implements evaluator-optimizer loops for **ShopCo Support Hub** (policy-compliant replies) and **ReleaseFlow** (release notes quality) in plain Python — rubrics, scoring, revision, and iteration budgets.

In [ ]:
"""
Evaluator-Optimizer Pattern Lab — ShopCo + ReleaseFlow.
"""

import json
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

USE_LIVE_LLM = False
DEFAULT_PASS_SCORE = 0.85
DEFAULT_MAX_ITERATIONS = 4


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"order_id": "ORD-1001", "customer": "alice@shop.com", "status": "shipped", "total_usd": 1299.0},
    "ORD-1002": {"order_id": "ORD-1002", "customer": "bob@shop.com", "status": "processing", "total_usd": 449.0},
}

POLICIES: dict[str, str] = {
    "returns": "30-day returns on unused items. [pol-returns-01]",
    "refunds": "Cite policy before promising refunds. Escalate over $1000. [pol-refunds-03]",
    "tone": "Professional, empathetic, no blame language.",
}

RELEASE_CHECKS: dict[str, dict[str, str]] = {
    "unit_tests": {"status": "pass", "detail": "412 tests passed"},
    "security_scan": {"status": "warn", "detail": "1 medium CVE in dev dependency"},
    "lint": {"status": "pass", "detail": "no issues"},
}

EVAL_TRACE: list[dict[str, Any]] = []

print("Evaluator-optimizer lab ready")

---

## 1. Evaluator-Optimizer vs Chain vs Single Shot

| Pattern | Iterations | Who decides to continue? |
|---------|------------|--------------------------|
| **Single shot** | 1 | Never |
| **Prompt chain** | Fixed N steps | Developer at build time |
| **Evaluator-optimizer** | Variable (1–N) | Evaluator score / rubric |

The evaluator acts as a **quality gate** with structured feedback — not just "try again" but "add policy citation" or "mention security warning."

---

## 2. Core Types — Draft, Evaluation, Run

In [ ]:
@dataclass
class Draft:
    content: str
    version: int = 1
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass
class CriterionScore:
    name: str
    score: float  # 0.0 – 1.0
    passed: bool
    feedback: str


@dataclass
class Evaluation:
    scores: list[CriterionScore]
    overall: float
    passed: bool
    revision_hints: list[str] = field(default_factory=list)


@dataclass
class EvalOptRun:
    run_id: str
    task: str
    drafts: list[Draft] = field(default_factory=list)
    evaluations: list[Evaluation] = field(default_factory=list)
    final_draft: Draft | None = None
    iterations: int = 0
    stopped_reason: str = ""


print("Core types ready")

---

## 3. ShopCo Generator — Draft Support Replies

The generator produces an initial reply from ticket context. Revision rounds append missing elements based on evaluator hints.

In [ ]:
def extract_order_id(text: str) -> str | None:
    m = re.search(r"ORD-\d+", text.upper())
    return m.group() if m else None


class ShopCoReplyGenerator:
    """Simulates LLM draft generation with deterministic revision behavior."""

    def generate(self, ticket: str, context: dict[str, Any], *, hints: list[str] | None = None) -> Draft:
        oid = context.get("order_id") or extract_order_id(ticket)
        order = ORDERS.get(oid) if oid else None
        hints = hints or []
        parts: list[str] = []

        if order:
            parts.append(f"Your order {oid} is currently {order['status']}.")
        else:
            parts.append("Thanks for contacting ShopCo.")

        if "refund" in ticket.lower():
            parts.append("We can help with your refund request.")

        draft = " ".join(parts)

        if any("policy citation" in h.lower() or "pol-" in h.lower() for h in hints):
            draft += " " + POLICIES["refunds"]
        if any("empathy" in h.lower() or "tone" in h.lower() for h in hints):
            draft = "We understand your frustration. " + draft
        if any("escalat" in h.lower() for h in hints) and order and order["total_usd"] > 1000:
            draft += " A specialist will follow up within 24 hours."
        if any("order amount" in h.lower() or "total" in h.lower() for h in hints) and order:
            draft += f" Order total: ${order['total_usd']:.0f}."

        version = context.get("_version", 1)
        return Draft(draft, version=version, metadata={"order_id": oid})


gen = ShopCoReplyGenerator()
sample = gen.generate("Refund for ORD-1001", {"order_id": "ORD-1001"})
print(f"Initial draft: {sample.content}")

---

## 4. ShopCo Evaluator — Rubric Scoring

In [ ]:
class ShopCoReplyEvaluator:
    """Rule-based evaluator — production may use LLM-as-judge or human review."""

    def __init__(self, pass_score: float = DEFAULT_PASS_SCORE):
        self.pass_score = pass_score

    def evaluate(self, draft: Draft, ticket: str, context: dict[str, Any]) -> Evaluation:
        text = draft.content
        oid = context.get("order_id") or extract_order_id(ticket)
        order = ORDERS.get(oid) if oid else None
        is_refund = "refund" in ticket.lower()
        scores: list[CriterionScore] = []
        hints: list[str] = []

        # Criterion: order reference
        has_order = oid and oid in text
        scores.append(CriterionScore(
            "order_reference", 1.0 if has_order else 0.3, has_order,
            "Include order ID." if not has_order else "Order referenced.",
        ))
        if not has_order and oid:
            hints.append(f"Mention order {oid} explicitly.")

        # Criterion: policy citation for refunds
        has_policy = "[pol-" in text or "pol-refunds" in text
        need_policy = is_refund
        pol_score = 1.0 if (not need_policy or has_policy) else 0.2
        scores.append(CriterionScore(
            "policy_citation", pol_score, pol_score >= 0.8,
            "Add policy citation [pol-refunds-03]." if need_policy and not has_policy else "Policy cited.",
        ))
        if need_policy and not has_policy:
            hints.append("Add policy citation from refunds policy.")

        # Criterion: empathy / tone
        empathetic = any(w in text.lower() for w in ("understand", "sorry", "apologize", "frustration"))
        angry_ticket = any(w in ticket.lower() for w in ("unacceptable", "angry", "terrible", "!!!"))
        tone_ok = empathetic or not angry_ticket
        scores.append(CriterionScore(
            "empathy", 1.0 if tone_ok else 0.4, tone_ok,
            "Add empathetic opening for upset customer." if not tone_ok else "Tone acceptable.",
        ))
        if not tone_ok:
            hints.append("Add empathy — acknowledge customer frustration.")

        # Criterion: high-value escalation
        needs_esc = order and order["total_usd"] > 1000 and is_refund
        has_esc = "specialist" in text.lower() or "follow up" in text.lower()
        esc_score = 1.0 if (not needs_esc or has_esc) else 0.5
        scores.append(CriterionScore(
            "escalation", esc_score, esc_score >= 0.8,
            "Mention specialist follow-up for orders over $1000." if needs_esc and not has_esc else "Escalation OK.",
        ))
        if needs_esc and not has_esc:
            hints.append("Add escalation note for high-value refund.")

        overall = sum(s.score for s in scores) / len(scores)
        return Evaluation(scores, overall, overall >= self.pass_score, hints)


evaluator = ShopCoReplyEvaluator()
ev = evaluator.evaluate(sample, "Refund for ORD-1001!!!", {"order_id": "ORD-1001"})
print(f"Score {ev.overall:.2f} passed={ev.passed} hints={ev.revision_hints}")

---

## 5. Evaluator-Optimizer Loop

In [ ]:
class EvaluatorOptimizer:
    def __init__(
        self,
        generator: ShopCoReplyGenerator,
        evaluator: ShopCoReplyEvaluator,
        *,
        max_iterations: int = DEFAULT_MAX_ITERATIONS,
        pass_score: float = DEFAULT_PASS_SCORE,
    ):
        self.generator = generator
        self.evaluator = evaluator
        self.max_iterations = max_iterations
        self.pass_score = pass_score

    def run(self, ticket: str, context: dict[str, Any] | None = None) -> EvalOptRun:
        ctx = dict(context or {})
        ctx.setdefault("order_id", extract_order_id(ticket))
        run = EvalOptRun(str(uuid.uuid4())[:8], ticket)
        hints: list[str] = []

        for i in range(1, self.max_iterations + 1):
            ctx["_version"] = i
            draft = self.generator.generate(ticket, ctx, hints=hints)
            run.drafts.append(draft)
            evaluation = self.evaluator.evaluate(draft, ticket, ctx)
            run.evaluations.append(evaluation)
            run.iterations = i

            if evaluation.passed:
                run.final_draft = draft
                run.stopped_reason = "passed"
                break
            hints = evaluation.revision_hints
            if i == self.max_iterations:
                run.final_draft = draft
                run.stopped_reason = "max_iterations"

        EVAL_TRACE.append({
            "run_id": run.run_id, "iterations": run.iterations,
            "stopped": run.stopped_reason,
            "final_score": run.evaluations[-1].overall if run.evaluations else 0,
            "ts": utcnow(),
        })
        return run


loop = EvaluatorOptimizer(gen, evaluator)
print("Evaluator-optimizer loop ready")

---

## 6. ShopCo End-to-End — Refund Ticket Refinement

In [ ]:
ticket = "I want a refund for ORD-1001 — this is unacceptable!!!"
run = loop.run(ticket)

print(f"Iterations: {run.iterations} ({run.stopped_reason})")
for i, (d, e) in enumerate(zip(run.drafts, run.evaluations), 1):
    print(f"\n--- v{i} score={e.overall:.2f} passed={e.passed} ---")
    print(d.content[:200] + ("..." if len(d.content) > 200 else ""))
    if e.revision_hints:
        print(f"Hints: {e.revision_hints}")

print(f"\nFinal reply:\n{run.final_draft.content if run.final_draft else 'none'}")

---

## 7. Simple Ticket — Early Exit on First Pass

In [ ]:
easy = loop.run("Where is ORD-1002?")
print(f"Simple ticket: {easy.iterations} iteration(s), score={easy.evaluations[-1].overall:.2f}")
print(easy.final_draft.content if easy.final_draft else "")

---

## 8. Strict vs Lenient Evaluators

In [ ]:
strict = EvaluatorOptimizer(gen, ShopCoReplyEvaluator(pass_score=0.95), max_iterations=3)
lenient = EvaluatorOptimizer(gen, ShopCoReplyEvaluator(pass_score=0.70), max_iterations=3)

t = "Refund ORD-1001 please"
rs = strict.run(t)
rl = lenient.run(t)
print(f"Strict:  {rs.iterations} iter, score={rs.evaluations[-1].overall:.2f}, reason={rs.stopped_reason}")
print(f"Lenient: {rl.iterations} iter, score={rl.evaluations[-1].overall:.2f}, reason={rl.stopped_reason}")

---

## 9. ReleaseFlow — Release Notes Evaluator-Optimizer

Draft release notes must include version, test summary, and security status before ship.

In [ ]:
class ReleaseNotesGenerator:
    def generate(self, version: str, checks: dict[str, dict[str, str]], *, hints: list[str] | None = None) -> Draft:
        hints = hints or []
        hint_blob = " ".join(hints).lower()
        title = f"# {version} Release Notes" if "version" in hint_blob else f"Release {version}"
        lines = [title, f"Tests: {checks['unit_tests']['detail']}."]
        if "security" in hint_blob or "cve" in hint_blob:
            sec = checks["security_scan"]
            lines.append(f"Security: {sec['status']} — {sec['detail']}.")
        return Draft("\n".join(lines), metadata={"version": version})


class ReleaseNotesEvaluator:
    def __init__(self, pass_score: float = 0.85):
        self.pass_score = pass_score

    def evaluate(self, draft: Draft, version: str, checks: dict[str, dict[str, str]]) -> Evaluation:
        text = draft.content
        scores: list[CriterionScore] = []
        hints: list[str] = []

        has_ver = version in text
        scores.append(CriterionScore("version", 1.0 if has_ver else 0.0, has_ver, "Include version."))
        if not has_ver:
            hints.append("Include version prominently.")

        has_tests = "test" in text.lower() or "412" in text
        scores.append(CriterionScore("tests", 1.0 if has_tests else 0.3, has_tests, "Add test summary."))
        if not has_tests:
            hints.append("Add unit test summary.")

        sec_status = checks["security_scan"]["status"]
        has_sec = "security" in text.lower() or "cve" in text.lower()
        need_sec = sec_status != "pass"
        sec_score = 1.0 if (not need_sec or has_sec) else 0.2
        scores.append(CriterionScore("security", sec_score, sec_score >= 0.8, "Document security scan result."))
        if need_sec and not has_sec:
            hints.append("Add security scan warning for medium CVE.")

        overall = sum(s.score for s in scores) / len(scores)
        return Evaluation(scores, overall, overall >= self.pass_score, hints)


class ReleaseEvalOpt:
    def __init__(self, max_iterations: int = 4):
        self.gen = ReleaseNotesGenerator()
        self.eval = ReleaseNotesEvaluator()
        self.max_iterations = max_iterations

    def run(self, version: str) -> EvalOptRun:
        run = EvalOptRun(str(uuid.uuid4())[:8], f"Release notes {version}")
        hints: list[str] = []
        for i in range(1, self.max_iterations + 1):
            draft = self.gen.generate(version, RELEASE_CHECKS, hints=hints)
            draft.version = i
            run.drafts.append(draft)
            ev = self.eval.evaluate(draft, version, RELEASE_CHECKS)
            run.evaluations.append(ev)
            run.iterations = i
            if ev.passed:
                run.final_draft = draft
                run.stopped_reason = "passed"
                break
            hints = ev.revision_hints
            if i == self.max_iterations:
                run.final_draft = draft
                run.stopped_reason = "max_iterations"
        return run


rf_run = ReleaseEvalOpt().run("v2.4.1")
print(f"Release notes: {rf_run.iterations} iterations ({rf_run.stopped_reason})")
print(rf_run.final_draft.content if rf_run.final_draft else "")

---

## 10. Iteration Budget and Cost Control

In [ ]:
@dataclass
class EvalOptBudget:
    max_iterations: int = 3
    max_tokens_estimate: int = 8000
    cost_per_iteration: int = 500  # simulated token units


class BudgetedEvalOpt(EvaluatorOptimizer):
    def __init__(self, generator, evaluator, budget: EvalOptBudget, **kw):
        super().__init__(generator, evaluator, max_iterations=budget.max_iterations, **kw)
        self.budget = budget

    def run(self, ticket: str, context: dict[str, Any] | None = None) -> EvalOptRun:
        run = super().run(ticket, context)
        estimated = run.iterations * self.budget.cost_per_iteration
        run.stopped_reason += f" (est. {estimated} token units)"
        return run


budgeted = BudgetedEvalOpt(gen, evaluator, EvalOptBudget(max_iterations=2))
br = budgeted.run("Refund ORD-1001 — terrible service!!!")
print(f"Budget cap 2: stopped={br.stopped_reason}, score={br.evaluations[-1].overall:.2f}")

---

## 11. Evaluator Feedback Quality

In [ ]:
def feedback_quality(evaluation: Evaluation) -> dict[str, Any]:
    """Good hints are specific and actionable — not vague 'improve quality'."""
    vague = {"better", "improve", "fix it", "try again"}
    specific = [h for h in evaluation.revision_hints if not any(v in h.lower() for v in vague)]
    return {
        "hint_count": len(evaluation.revision_hints),
        "specific_hints": specific,
        "failed_criteria": [s.name for s in evaluation.scores if not s.passed],
    }


for e in run.evaluations:
    fq = feedback_quality(e)
    print(f"Score {e.overall:.2f}: failed={fq['failed_criteria']} hints={fq['specific_hints']}")

---

## 12. Trace Inspection

In [ ]:
print(f"Eval-opt runs logged: {len(EVAL_TRACE)}")
for entry in EVAL_TRACE[-5:]:
    print(f"  {entry['run_id']}: {entry['iterations']} iter, score={entry['final_score']:.2f}, {entry['stopped']}")

---

## 13. When to Use Evaluator-Optimizer

| Good fit | Poor fit |
|----------|----------|
| Subjective quality (tone, completeness) | Deterministic transforms with one correct answer |
| Clear rubric or checklist | No way to score drafts reliably |
| High cost of bad output (policy, compliance) | Latency-sensitive paths needing one shot |
| Release notes, emails, summaries | Simple classification or lookup |

---

## 14. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| Vague evaluator ("not good enough") | Optimizer cannot revise | Rubric + specific hints |
| Same model for gen and eval without separation | Blind spots, grade inflation | Separate prompts or rule checks |
| Infinite loop | Cost blow-up | `max_iterations` + budget |
| Evaluator too strict | Never passes, wasted tokens | Calibrate pass threshold on samples |
| Evaluator too lenient | Ships bad output | Add hard rule checks (policy IDs) |
| No trace of iterations | Cannot debug quality regressions | Log drafts, scores, hints |

---

## 15. Check Your Understanding

1. How is evaluator-optimizer different from a fixed **prompt chain**?
2. What three roles appear in the loop (generate, evaluate, optimize)?
3. Why must evaluator feedback be **specific**?
4. When does ShopCo require a **policy citation**?
5. What stops the loop if quality never reaches the threshold?
6. Why might you use a **stricter** pass score for refund replies than FAQs?
7. What ReleaseFlow criterion fails when security scan has a warning?

---

## 16. Optional — LLM Evaluator-Optimizer

Set `USE_LIVE_LLM = True` to use a model for generation and a separate judge pass.

In [ ]:
if USE_LIVE_LLM:
    import os
    from openai import OpenAI

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    ticket = "Refund ORD-1001"
    draft_resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Draft ShopCo support reply: {ticket}"}],
    )
    draft_text = draft_resp.choices[0].message.content or ""
    judge_resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Score 0-1 and list missing rubric items as JSON."},
            {"role": "user", "content": draft_text},
        ],
        response_format={"type": "json_object"},
    )
    print(judge_resp.choices[0].message.content)
else:
    print("Offline mode — rule-based evaluator-optimizer for ShopCo and ReleaseFlow.")

---

## 17. Summary

Evaluator-optimizer improves output through **conditional iteration**:

- **Generate** a draft from task context
- **Evaluate** against a rubric with scores and revision hints
- **Optimize** by regenerating with hints until pass or budget exhausted
- **ShopCo** — policy citations, empathy, escalation for high-value refunds
- **ReleaseFlow** — release notes with version, tests, and security disclosure

Invest in evaluator quality and iteration caps; vague feedback wastes tokens without improving drafts.